In [77]:
import numpy as np

class Tensor:

    def __init__(self, data, children=()):
        self.data = data
        self.children = set(children)
        self.grad = np.zeros(data.shape)
        self.backwards = lambda: None

    def __add__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(np.array([other]))
        out = Tensor(self.data + other.data, (self, other))

        def backwards():
            self.grad += unbroadcast(out.grad, self.data.shape)
            other.grad += unbroadcast(out.grad, other.data.shape)

        out.backwards = backwards
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(np.array([other]))
        out = Tensor(self.data * other.data, (self, other))

        def backwards():
            self.grad += unbroadcast(out.grad * other.data, self.data.shape)
            other.grad += unbroadcast(out.grad * self.data, other.data.shape)

        out.backwards = backwards
        return out

    def __matmul__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(np.array([other]))
        out = Tensor(self.data @ other.data, (self, other))

        def backwards():
            self.grad += out.grad @ other.data.T
            other.grad += self.data.T @ out.grad

        out.backwards = backwards
        return out

    def __pow__(self, other):
        out = Tensor(self.data ** other, (self, ))

        def backwards():
            self.grad += other * (self.data ** (other - 1)) * out.grad

        out.backwards = backwards
        return out

    def __truediv__(self, other):
        return self * (other ** -1)

    def __sub__(self, other):
        return self + (other * -1)

    def tanh(self):
        x = self.data
        t = (np.exp(2*x) - 1) / (np.exp(2*x) + 1)
        out = Tensor(t, (self, ))

        def backwards():
            self.grad += (1 - t**2) * out.grad

        out.backwards = backwards
        return out

    def sum(self, axis=0):
        out = Tensor(self.data.sum(axis=axis), (self, ))

        def backwards():
            self.grad += np.ones_like(self.data) * np.expand_dims(out.grad, axis=axis)

        out.backwards = backwards
        return out

    def __repr__(self):
        return f'Data:\n{self.data}\nGrad:\n{self.grad}'

    def backward(self):
        topo = []
        visited = set()

        def topo_sort(node):
            if node not in visited:
                visited.add(node)
                for child in node.children:
                    topo_sort(child)
                topo.append(node)

        self.grad = np.ones(self.data.shape)
        topo_sort(self)

        for node in reversed(topo):
            node.backwards()

def unbroadcast(grad, target_shape):
    while grad.ndim > len(target_shape):
        grad = grad.sum(axis=0)
    for axis, size in enumerate(target_shape):
        if size == 1:
            grad = grad.sum(axis=axis, keepdims=True)
    return grad

def zero_grad(model):
    for p in model.parameters:
        p.grad = np.zeros(p.data.shape)

def model_step(model, learning_rate):
    for p in model.parameters:
        p.data += -learning_rate * p.grad

def MSELoss(target, prediction):
    return ((target - prediction) ** 2).sum(axis=1).sum(axis=0)

class Layer:

    def __init__(self, nin, nout):
        self.weights = Tensor(np.random.uniform(-1, 1, (nin, nout)))
        self.bias = Tensor(np.random.uniform(-1, 1, nout))

    def __call__(self, x):
        act = x @ self.weights + self.bias
        return act.tanh()

class MLP:

    def __init__(self, nin, nouts):
        size = [nin] + nouts
        self.layers = [Layer(size[i], size[i + 1]) for i in range(len(nouts))]
        self.parameters = []
        for layer in self.layers:
            self.parameters.append(layer.weights)
            self.parameters.append(layer.bias)

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

model = MLP(784, [16, 16, 10])

model.parameters


[Data:
 [[ 0.16247104  0.4072776   0.52031735 ...  0.6181759   0.89488499
    0.3035894 ]
  [ 0.98781935 -0.58056548  0.07101103 ... -0.98487923 -0.69516204
   -0.13332875]
  [-0.67637354  0.74496115  0.5672083  ... -0.22806243 -0.61255278
    0.76161912]
  ...
  [-0.93181491 -0.68485582 -0.40955357 ...  0.04100114 -0.0472592
   -0.13112278]
  [ 0.83157009 -0.13805018 -0.3845619  ... -0.25675034  0.72493471
    0.80644307]
  [-0.77856261  0.16759843 -0.93843551 ...  0.07787808  0.0896691
   -0.38618863]]
 Grad:
 [[0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  ...
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]],
 Data:
 [ 0.10539476 -0.41957713  0.87296095 -0.38162783 -0.54196092  0.1088247
   0.22523928  0.88343254 -0.3617318  -0.38014009  0.94899235  0.19788976
   0.95694493 -0.61129108 -0.88927344  0.832127  ]
 Grad:
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.],
 Data:
 [[-0.12666553  0.18985716  0.66843067  0.946003

In [78]:
from torchvision import datasets

train_data = datasets.MNIST('../data', download=True, train=True)
test_data = datasets.MNIST('../data', download=True, train=False)

X_train = []
Y_train = []

X_test = []
Y_test = []

for image, label in train_data:
    image = [p / 255 for p in image.get_flattened_data()]
    
    Y_one_hot = [0.0 for _ in range(10)]
    Y_one_hot[label] = 1.0
    
    X_train.append(image)
    Y_train.append(Y_one_hot)

for image, label in test_data:
    image = [p / 255 for p in image.get_flattened_data()]
    
    Y_one_hot = [0.0 for _ in range(10)]
    Y_one_hot[label] = 1.0
    
    X_test.append(image)
    Y_test.append(Y_one_hot)

len(X_train), len(Y_train), len(X_test), len(Y_test)

(60000, 60000, 10000, 10000)

In [83]:
train_loader = []
test_loader = []

batch_size = 50

for i in range(0, len(X_train), batch_size):
    X_batch = Tensor(np.array(X_train[i:i + batch_size]))
    Y_batch = Tensor(np.array(Y_train[i:i + batch_size]))
    train_loader.append([X_batch, Y_batch])

for i in range(0, len(X_test), batch_size):
    X_batch = Tensor(np.array(X_test[i:i + batch_size]))
    Y_batch = Tensor(np.array(Y_test[i:i + batch_size]))
    test_loader.append([X_batch, Y_batch])

len(train_loader), len(test_loader)


(1200, 200)

In [84]:
epochs = 100
learning_rate = 0.004

model = MLP(784, [16, 16, 10])

for epoch in range(epochs):

    for batch, (X, targets) in enumerate(train_loader):
    
        predictions = model(X)
    
        loss = MSELoss(targets, predictions)
    
        zero_grad(model)
    
        loss.backward()
    
        model_step(model, learning_rate)

    if epoch % 10 == 0:
        print(f'Loss: {loss.data:.4f}')
    

Loss: 0.9663
Loss: 0.9509
Loss: 0.9435
Loss: 0.9449
Loss: 0.9217
Loss: 0.9210
Loss: 0.9184
Loss: 0.9340
Loss: 0.9075
Loss: 0.9226


In [105]:
total = 0
correct = 0

for X, targets in test_loader:
    
        predictions = model(X)

        for batch in range(batch_size):
            
            total += 1

            prediction = np.argmax(predictions.data[batch])

            target = np.argmax(targets.data[batch])
    
            if prediction == target:
                correct += 1

print(f'Final Result: {correct} / {total} = {(correct/total*100):.2f}%')


Final Result: 9228 / 10000 = 92.28%
